# OfficeQA Retrieval-Only Colab Runner

This notebook is the primary Colab workflow for the project. It handles:

1. mounting Google Drive
2. cloning or updating the repo
3. downloading missing Treasury Bulletin PDFs
4. preparing the page manifest and BM25 index
5. building multimodal FAISS indexes for CLIP and SigLIP
6. running all retrieval experiments
7. exporting plots and CSVs for the final report

Use `RUN_MODE = "smoke"` first to validate the pipeline quickly, then switch to `RUN_MODE = "full"` for the final benchmark.


## Before You Run

For `RUN_MODE = "full"`, make sure `officeqa_pro.csv` is already in:

`/content/drive/MyDrive/officeqa/officeqa_pro.csv`

The notebook will download missing PDFs into:

`/content/drive/MyDrive/officeqa/pdfs/`

For `RUN_MODE = "smoke"`, the notebook uses the bundled smoke CSV from the repo, so you do not need to upload a separate smoke file.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/hungngo2/cs445-final-project-retrieval.git'
REPO_DIR = Path('/content/officeqa-retrieval')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml,faiss,dev]'], check=True)
subprocess.run(['nvidia-smi'], check=False)

print(f'Repo ready at {REPO_DIR}')


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RUN_MODE = 'smoke'  # change to 'full' for the real benchmark
RUN_TAG = 'officeqa_smoke_faiss' if RUN_MODE == 'smoke' else 'officeqa_pro_faiss'
TOP_K = 50
BATCH_SIZE = 8
PAGE_BATCH_SIZE = 16 if RUN_MODE == 'smoke' else 32
DPI = 150
FORCE_REBUILD = False
DOWNLOAD_MISSING_PDFS = True

OFFICEQA_ROOT = Path('/content/drive/MyDrive/officeqa')
PDF_DIR = OFFICEQA_ROOT / 'pdfs'
ARTIFACT_DIR = OFFICEQA_ROOT / 'artifacts' / RUN_TAG
RESULTS_DIR = OFFICEQA_ROOT / 'results' / RUN_TAG
RENDER_CACHE = ARTIFACT_DIR / 'render_cache'
ANALYSIS_DIR = RESULTS_DIR / 'analysis'

QUESTIONS_CSV = REPO_DIR / 'data' / 'officeqa_pro_smoke.csv' if RUN_MODE == 'smoke' else OFFICEQA_ROOT / 'officeqa_pro.csv'
if not QUESTIONS_CSV.exists():
    raise FileNotFoundError(f'Missing questions CSV: {QUESTIONS_CSV}')

for path in [PDF_DIR, ARTIFACT_DIR, RESULTS_DIR, ANALYSIS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BM25_INDEX_PATH = ARTIFACT_DIR / 'page_bm25.pkl'
MANIFEST_PATH = ARTIFACT_DIR / 'page_manifest.jsonl'
SANITY_PATH = ARTIFACT_DIR / 'sanity_questions.json'
SUMMARY_PATH = RESULTS_DIR / 'summary.csv'

EXPERIMENTS = [
    {
        'run_name': 'bm25',
        'eval_model': 'bm25',
        'index_path': BM25_INDEX_PATH,
        'index_model': None,
        'crop_mode': None,
    },
    {
        'run_name': 'clip_faiss_full',
        'eval_model': 'clip_faiss',
        'index_path': ARTIFACT_DIR / 'clip_faiss_full',
        'index_model': 'clip',
        'crop_mode': 'full',
    },
    {
        'run_name': 'clip_faiss_fixed',
        'eval_model': 'clip_faiss',
        'index_path': ARTIFACT_DIR / 'clip_faiss_fixed',
        'index_model': 'clip',
        'crop_mode': 'fixed_2x2',
    },
    {
        'run_name': 'siglip_faiss_full',
        'eval_model': 'siglip_faiss',
        'index_path': ARTIFACT_DIR / 'siglip_faiss_full',
        'index_model': 'siglip',
        'crop_mode': 'full',
    },
    {
        'run_name': 'siglip_faiss_fixed',
        'eval_model': 'siglip_faiss',
        'index_path': ARTIFACT_DIR / 'siglip_faiss_fixed',
        'index_model': 'siglip',
        'crop_mode': 'fixed_2x2',
    },
]


def run_cmd(args, cwd=REPO_DIR, env=None):
    args = [str(arg) for arg in args]
    print('$ ' + ' '.join(shlex.quote(arg) for arg in args))
    subprocess.run(args, cwd=str(cwd), check=True, env={**os.environ, **(env or {})})


def index_ready(index_dir: Path) -> bool:
    return all((index_dir / name).exists() for name in ['index.faiss', 'metadata.jsonl', 'config.json'])


display(Markdown(
    f'''**Run config**

- `RUN_MODE`: `{RUN_MODE}`
- `QUESTIONS_CSV`: `{QUESTIONS_CSV}`
- `PDF_DIR`: `{PDF_DIR}`
- `ARTIFACT_DIR`: `{ARTIFACT_DIR}`
- `RESULTS_DIR`: `{RESULTS_DIR}`
- `TOP_K`: `{TOP_K}`
- `FORCE_REBUILD`: `{FORCE_REBUILD}`
    '''
))


In [ ]:
questions_df = pd.read_csv(QUESTIONS_CSV)
required_stems = sorted({Path(token).stem for value in questions_df['source_files'].fillna('') for token in str(value).split() if token})
existing_pdf_count = sum((PDF_DIR / f'{stem}.pdf').exists() for stem in required_stems)

print(f'Questions: {len(questions_df)}')
print(f'Unique required PDFs: {len(required_stems)}')
print(f'PDFs already present in Drive: {existing_pdf_count}')

display(questions_df[['uid', 'difficulty', 'question', 'source_files', 'source_docs']].head(10))


In [ ]:
if DOWNLOAD_MISSING_PDFS:
    run_cmd([
        sys.executable,
        'scripts/download_officeqa_pdfs.py',
        '--questions-csv', QUESTIONS_CSV,
        '--pdf-dir', PDF_DIR,
        '--skip-existing',
    ])

if FORCE_REBUILD or not MANIFEST_PATH.exists():
    run_cmd([
        sys.executable,
        'scripts/prepare_data.py',
        '--questions-csv', QUESTIONS_CSV,
        '--pdf-dir', PDF_DIR,
        '--manifest-out', MANIFEST_PATH,
        '--sanity-out', SANITY_PATH,
    ])
else:
    print(f'Skipping manifest build because {MANIFEST_PATH} already exists.')

if FORCE_REBUILD or not BM25_INDEX_PATH.exists():
    run_cmd([
        sys.executable,
        'scripts/build_page_index.py',
        '--manifest', MANIFEST_PATH,
        '--index-out', BM25_INDEX_PATH,
    ])
else:
    print(f'Skipping BM25 build because {BM25_INDEX_PATH} already exists.')


## Build Multimodal Indexes Incrementally

Run the next code cells one at a time. Each cell only builds one FAISS index, so Colab can checkpoint progress and you can stop after any completed build.


In [ ]:
from time import time

INDEX_RUNS = [experiment for experiment in EXPERIMENTS if experiment['eval_model'] != 'bm25']
INDEX_RUN_BY_NAME = {experiment['run_name']: experiment for experiment in INDEX_RUNS}
INDEX_SEQUENCE = [experiment['run_name'] for experiment in INDEX_RUNS]


def index_status_df():
    rows = []
    for position, run_name in enumerate(INDEX_SEQUENCE, start=1):
        experiment = INDEX_RUN_BY_NAME[run_name]
        index_dir = experiment['index_path']
        rows.append({
            'order': position,
            'run_name': run_name,
            'model': experiment['index_model'],
            'crop_mode': experiment['crop_mode'],
            'index_dir': str(index_dir),
            'ready': index_ready(index_dir),
            'has_index_faiss': (index_dir / 'index.faiss').exists(),
            'has_metadata': (index_dir / 'metadata.jsonl').exists(),
            'has_config': (index_dir / 'config.json').exists(),
        })
    return pd.DataFrame(rows)


def show_index_status():
    display(index_status_df())


def build_index(run_name: str):
    experiment = INDEX_RUN_BY_NAME[run_name]
    index_dir = experiment['index_path']
    ordinal = INDEX_SEQUENCE.index(run_name) + 1
    total = len(INDEX_SEQUENCE)

    print('=' * 80)
    print(f'Building index {ordinal}/{total}: {run_name}')
    print(f'Model: {experiment["index_model"]}')
    print(f'Crop mode: {experiment["crop_mode"]}')
    print(f'Index dir: {index_dir}')
    print('=' * 80)

    if not FORCE_REBUILD and index_ready(index_dir):
        print(f'Skipping {run_name} because the index already exists at {index_dir}.')
        return

    started_at = time()
    run_cmd([
        sys.executable,
        'scripts/build_multimodal_index.py',
        '--manifest', MANIFEST_PATH,
        '--index-dir', index_dir,
        '--model', experiment['index_model'],
        '--crop-mode', experiment['crop_mode'],
        '--render-cache', RENDER_CACHE,
        '--batch-size', str(BATCH_SIZE),
        '--page-batch-size', str(PAGE_BATCH_SIZE),
        '--dpi', str(DPI),
    ])
    elapsed_minutes = (time() - started_at) / 60.0
    print(f'Finished {run_name} in {elapsed_minutes:.2f} minutes.')


show_index_status()


In [ ]:
build_index('clip_faiss_full')
show_index_status()


In [ ]:
build_index('clip_faiss_fixed')
show_index_status()


In [ ]:
build_index('siglip_faiss_full')
show_index_status()


In [ ]:
build_index('siglip_faiss_fixed')
show_index_status()


## Run Experiments Incrementally

Run the next code cells one at a time. Each cell only launches one experiment, so Colab can checkpoint progress and you can stop after any run if needed.


In [ ]:
from time import time

EXPERIMENT_BY_NAME = {experiment['run_name']: experiment for experiment in EXPERIMENTS}
RUN_SEQUENCE = [experiment['run_name'] for experiment in EXPERIMENTS]


def experiment_status_df():
    rows = []
    for position, run_name in enumerate(RUN_SEQUENCE, start=1):
        experiment = EXPERIMENT_BY_NAME[run_name]
        metrics_path = RESULTS_DIR / run_name / 'metrics.json'
        rows.append({
            'order': position,
            'run_name': run_name,
            'model': experiment['eval_model'],
            'index_path': str(experiment['index_path']),
            'done': metrics_path.exists(),
            'metrics_path': str(metrics_path),
        })
    return pd.DataFrame(rows)


def show_experiment_status():
    display(experiment_status_df())


def run_experiment(run_name: str):
    experiment = EXPERIMENT_BY_NAME[run_name]
    out_dir = RESULTS_DIR / experiment['run_name']
    metrics_path = out_dir / 'metrics.json'
    ordinal = RUN_SEQUENCE.index(run_name) + 1
    total = len(RUN_SEQUENCE)

    print('=' * 80)
    print(f'Running experiment {ordinal}/{total}: {run_name}')
    print(f'Model: {experiment["eval_model"]}')
    print(f'Index: {experiment["index_path"]}')
    print(f'Output dir: {out_dir}')
    print('=' * 80)

    if metrics_path.exists() and not FORCE_REBUILD:
        print(f'Skipping {run_name} because metrics already exist at {metrics_path}.')
        return

    started_at = time()
    run_cmd([
        sys.executable,
        'scripts/run_retrieval_eval.py',
        '--questions-csv', QUESTIONS_CSV,
        '--index', experiment['index_path'],
        '--out-dir', out_dir,
        '--model', experiment['eval_model'],
        '--top-k', str(TOP_K),
    ])
    elapsed_minutes = (time() - started_at) / 60.0
    print(f'Finished {run_name} in {elapsed_minutes:.2f} minutes.')


def refresh_summary():
    run_cmd([
        sys.executable,
        'scripts/make_report_tables.py',
        '--results-dir', RESULTS_DIR,
        '--summary-out', SUMMARY_PATH,
    ])
    summary_df = pd.read_csv(SUMMARY_PATH).sort_values('page_mrr', ascending=False)
    display(summary_df)
    return summary_df


show_experiment_status()


In [ ]:
run_experiment('bm25')
show_experiment_status()


In [ ]:
run_experiment('clip_faiss_full')
show_experiment_status()


In [ ]:
run_experiment('clip_faiss_fixed')
show_experiment_status()


In [ ]:
run_experiment('siglip_faiss_full')
show_experiment_status()


In [ ]:
run_experiment('siglip_faiss_fixed')
show_experiment_status()


In [ ]:
summary_df = refresh_summary()


## Analysis Exports

The next cells create paper-friendly CSVs and plots under:

`/content/drive/MyDrive/officeqa/results/<RUN_TAG>/analysis/`


### Per-Run Analysis Exports

Use the next cells to export analysis artifacts for one run at a time. Each run writes into `analysis/runs/<run_name>/`, which is helpful when only some experiments have finished.


In [ ]:
import json
import matplotlib.pyplot as plt

RUN_ANALYSIS_DIR = ANALYSIS_DIR / 'runs'
RUN_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


def export_run_analysis(run_name: str, target_uid: str | None = None, top_n: int = 10):
    metrics_path = RESULTS_DIR / run_name / 'metrics.json'
    ranked_path = RESULTS_DIR / run_name / 'ranked_pages.jsonl'
    if not metrics_path.exists():
        raise FileNotFoundError(f'Metrics not found for {run_name}: {metrics_path}')
    if not ranked_path.exists():
        raise FileNotFoundError(f'Ranked predictions not found for {run_name}: {ranked_path}')

    run_dir = RUN_ANALYSIS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    metrics = json.loads(metrics_path.read_text())
    summary_df = pd.DataFrame([metrics['summary']])
    summary_df.insert(0, 'run_name', run_name)
    summary_df.to_csv(run_dir / 'summary.csv', index=False)
    display(summary_df)

    per_query_df = pd.DataFrame(metrics['per_query']).sort_values(['page_mrr', 'doc_mrr'], ascending=[False, False])
    per_query_df.to_csv(run_dir / 'per_query_metrics.csv', index=False)
    display(per_query_df.head(20))

    difficulty_cols = [column for column in ['page_mrr', 'doc_mrr', 'page_recall_at_1', 'page_recall_at_5', 'page_recall_at_10', 'doc_recall_at_1', 'doc_recall_at_5', 'doc_recall_at_10'] if column in per_query_df.columns]
    difficulty_summary = per_query_df.groupby('difficulty')[difficulty_cols].mean().reset_index()
    difficulty_summary.to_csv(run_dir / 'difficulty_summary.csv', index=False)
    display(difficulty_summary)

    ax = per_query_df.sort_values('uid').plot(
        x='uid',
        y='page_mrr',
        kind='bar',
        figsize=(12, 4),
        rot=45,
        legend=False,
        title=f'{run_name}: per-query page MRR',
    )
    ax.set_ylabel('page MRR')
    plt.tight_layout()
    plt.savefig(run_dir / 'per_query_page_mrr.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()

    ranked_rows = []
    with ranked_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            ranked_rows.append(json.loads(line))
    ranked_df = pd.DataFrame(ranked_rows)
    ranked_df.to_csv(run_dir / 'ranked_pages.csv', index=False)

    if target_uid is None:
        target_uid = questions_df['uid'].iloc[0]

    target_rows = ranked_df.loc[ranked_df['uid'] == target_uid].sort_values('rank').head(top_n)
    target_rows.to_csv(run_dir / f'qualitative_{target_uid}.csv', index=False)
    if not target_rows.empty:
        display(target_rows[['uid', 'rank', 'doc_id', 'page_num', 'score', 'bm25_score', 'matched_crop']])

    print(f'Saved run summary to {run_dir / "summary.csv"}')
    print(f'Saved per-query metrics to {run_dir / "per_query_metrics.csv"}')
    print(f'Saved difficulty summary to {run_dir / "difficulty_summary.csv"}')
    print(f'Saved ranked pages to {run_dir / "ranked_pages.csv"}')
    print(f'Saved per-query plot to {run_dir / "per_query_page_mrr.png"}')
    print(f'Saved qualitative slice to {run_dir / f"qualitative_{target_uid}.csv"}')


def export_all_completed_run_analyses():
    completed = [run_name for run_name in RUN_SEQUENCE if (RESULTS_DIR / run_name / 'metrics.json').exists()]
    print(f'Exporting per-run analysis for: {completed}')
    for run_name in completed:
        print(f'\n===== {run_name} =====')
        export_run_analysis(run_name)


In [ ]:
export_run_analysis('bm25')


In [ ]:
export_run_analysis('clip_faiss_full')


In [ ]:
export_run_analysis('clip_faiss_fixed')


In [ ]:
export_run_analysis('siglip_faiss_full')


In [ ]:
export_run_analysis('siglip_faiss_fixed')


In [ ]:
# Optional: export every run that has finished so far
export_all_completed_run_analyses()


In [ ]:
import matplotlib.pyplot as plt

summary_df = pd.read_csv(SUMMARY_PATH).sort_values('page_mrr', ascending=False)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(ANALYSIS_DIR / 'summary_table.csv', index=False)

page_cols = ['page_mrr', 'page_recall_at_1', 'page_recall_at_5', 'page_recall_at_10']
doc_cols = ['doc_mrr', 'doc_recall_at_1', 'doc_recall_at_5', 'doc_recall_at_10']

page_plot_df = summary_df.set_index('run_name')[page_cols]
doc_plot_df = summary_df.set_index('run_name')[doc_cols]

ax = page_plot_df.plot(kind='bar', figsize=(12, 5), rot=30, title='OfficeQA Page Retrieval Metrics by Run')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'summary_page_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

ax = doc_plot_df.plot(kind='bar', figsize=(12, 5), rot=30, title='OfficeQA Document Retrieval Metrics by Run')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'summary_doc_metrics.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

print(f'Saved summary table to {ANALYSIS_DIR / "summary_table.csv"}')
print(f'Saved page-metric plot to {ANALYSIS_DIR / "summary_page_metrics.png"}')
print(f'Saved doc-metric plot to {ANALYSIS_DIR / "summary_doc_metrics.png"}')


In [ ]:
import json

records = []
for metrics_path in sorted(RESULTS_DIR.glob('*/metrics.json')):
    data = json.loads(metrics_path.read_text())
    run_name = metrics_path.parent.name
    for row in data['per_query']:
        item = dict(row)
        item['run_name'] = run_name
        records.append(item)

per_query_df = pd.DataFrame(records)
per_query_df.to_csv(ANALYSIS_DIR / 'per_query_metrics.csv', index=False)
display(per_query_df.head(20))

pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0).sort_index()
display(pivot.head(20))

if len(pivot) <= 30:
    ax = pivot.plot(kind='bar', figsize=(14, 6), rot=45, title='Per-question Page MRR by Run')
    ax.set_ylabel('page MRR')
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'per_query_page_mrr.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print('Skipping per-question bar chart because there are more than 30 questions. Use the CSV export instead.')

best_run_by_query = per_query_df.sort_values(['uid', 'page_mrr', 'doc_mrr'], ascending=[True, False, False]).drop_duplicates('uid')
best_run_by_query.to_csv(ANALYSIS_DIR / 'best_run_by_query.csv', index=False)

difficulty_summary = (
    per_query_df.groupby(['run_name', 'difficulty'])[['page_mrr', 'doc_mrr', 'page_recall_at_10', 'doc_recall_at_10']]
    .mean()
    .reset_index()
)
difficulty_summary.to_csv(ANALYSIS_DIR / 'difficulty_summary.csv', index=False)
display(difficulty_summary)

print(f'Saved per-query CSV to {ANALYSIS_DIR / "per_query_metrics.csv"}')
print(f'Saved best-run-by-query CSV to {ANALYSIS_DIR / "best_run_by_query.csv"}')
print(f'Saved difficulty summary CSV to {ANALYSIS_DIR / "difficulty_summary.csv"}')


### Advanced Comparison Analysis

These cells generate stronger report-ready analyses from the saved rankings: recall curves, per-query deltas vs BM25, win/loss summaries, crop-benefit plots, and optional embedding-space visualizations.


In [ ]:
import json
from collections import defaultdict

import numpy as np
from officeqa_retrieval.dataset import load_questions

questions = load_questions(QUESTIONS_CSV)
question_by_uid = {question.uid: question for question in questions}
completed_runs = [run_name for run_name in RUN_SEQUENCE if (RESULTS_DIR / run_name / 'metrics.json').exists() and (RESULTS_DIR / run_name / 'ranked_pages.jsonl').exists()]
print('Completed runs:', completed_runs)


def gold_pages_for_uid(uid: str):
    question = question_by_uid[uid]
    return {
        (reference.doc_id, reference.page_num)
        for reference in question.gold_references
        if reference.doc_id and reference.page_num is not None
    }


def gold_docs_for_uid(uid: str):
    question = question_by_uid[uid]
    return {reference.doc_id for reference in question.gold_references if reference.doc_id}


def load_ranked_df(run_name: str):
    ranked_path = RESULTS_DIR / run_name / 'ranked_pages.jsonl'
    rows = []
    with ranked_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            row['run_name'] = run_name
            rows.append(row)
    ranked_df = pd.DataFrame(rows)
    if ranked_df.empty:
        return ranked_df
    ranked_df = ranked_df.sort_values(['uid', 'rank']).reset_index(drop=True)
    ranked_df['page_key'] = list(zip(ranked_df['doc_id'], ranked_df['page_num']))
    ranked_df['is_gold_page'] = ranked_df.apply(lambda row: row['page_key'] in gold_pages_for_uid(row['uid']), axis=1)
    ranked_df['is_gold_doc'] = ranked_df.apply(lambda row: row['doc_id'] in gold_docs_for_uid(row['uid']), axis=1)
    return ranked_df


def unique_doc_rank_rows(run_df: pd.DataFrame):
    seen = set()
    rows = []
    for _, row in run_df.sort_values('rank').iterrows():
        if row['doc_id'] in seen:
            continue
        seen.add(row['doc_id'])
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=run_df.columns)
    return pd.DataFrame(rows)


def first_hit_rank(series):
    hits = series.loc[series].index
    if len(hits) == 0:
        return np.nan
    return int(hits.min())


def build_first_hit_rank_df():
    rows = []
    for run_name in completed_runs:
        ranked_df = load_ranked_df(run_name)
        for uid, group in ranked_df.groupby('uid'):
            group = group.sort_values('rank').copy()
            group.index = group['rank']
            doc_group = unique_doc_rank_rows(group).copy()
            if not doc_group.empty:
                doc_group.index = range(1, len(doc_group) + 1)
            top_scores = group['score'].tolist()[:2]
            score_margin = top_scores[0] - top_scores[1] if len(top_scores) >= 2 else np.nan
            rows.append({
                'run_name': run_name,
                'uid': uid,
                'difficulty': question_by_uid[uid].difficulty,
                'gold_page_count': len(gold_pages_for_uid(uid)),
                'page_first_hit_rank': first_hit_rank(group['is_gold_page']),
                'doc_first_hit_rank': first_hit_rank(doc_group['is_gold_doc']) if not doc_group.empty else np.nan,
                'top1_score_margin': score_margin,
                'top1_is_gold_page': bool(group.loc[group['rank'] == 1, 'is_gold_page'].any()),
                'top1_is_gold_doc': bool(group.loc[group['rank'] == 1, 'is_gold_doc'].any()),
            })
    return pd.DataFrame(rows)


first_hit_rank_df = build_first_hit_rank_df()
first_hit_rank_df.to_csv(ANALYSIS_DIR / 'first_hit_ranks.csv', index=False)
display(first_hit_rank_df.head(20))


In [ ]:
ks = list(range(1, min(TOP_K, 50) + 1))
curve_rows = []
for run_name, group in first_hit_rank_df.groupby('run_name'):
    page_ranks = group['page_first_hit_rank']
    doc_ranks = group['doc_first_hit_rank']
    for k in ks:
        curve_rows.append({
            'run_name': run_name,
            'k': k,
            'page_recall': float((page_ranks <= k).fillna(False).mean()),
            'doc_recall': float((doc_ranks <= k).fillna(False).mean()),
        })

curve_df = pd.DataFrame(curve_rows)
curve_df.to_csv(ANALYSIS_DIR / 'recall_curves.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=False)
for run_name, group in curve_df.groupby('run_name'):
    axes[0].plot(group['k'], group['page_recall'], marker='o', label=run_name)
    axes[1].plot(group['k'], group['doc_recall'], marker='o', label=run_name)

axes[0].set_title('Page Recall@k Curve')
axes[0].set_xlabel('k')
axes[0].set_ylabel('recall')
axes[0].set_ylim(0, 1.05)
axes[1].set_title('Document Recall@k Curve')
axes[1].set_xlabel('k')
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'recall_curves.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

print(f'Saved recall curve data to {ANALYSIS_DIR / "recall_curves.csv"}')
print(f'Saved recall curve figure to {ANALYSIS_DIR / "recall_curves.png"}')


In [ ]:
page_mrr_pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr')
doc_mrr_pivot = per_query_df.pivot(index='uid', columns='run_name', values='doc_mrr')

if 'bm25' not in page_mrr_pivot.columns:
    raise ValueError('BM25 results are required for delta-vs-BM25 analysis.')

delta_rows = []
for run_name in completed_runs:
    if run_name == 'bm25':
        continue
    for uid in page_mrr_pivot.index:
        delta_rows.append({
            'uid': uid,
            'difficulty': question_by_uid[uid].difficulty,
            'run_name': run_name,
            'page_mrr_delta_vs_bm25': page_mrr_pivot.loc[uid, run_name] - page_mrr_pivot.loc[uid, 'bm25'],
            'doc_mrr_delta_vs_bm25': doc_mrr_pivot.loc[uid, run_name] - doc_mrr_pivot.loc[uid, 'bm25'],
        })

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(ANALYSIS_DIR / 'delta_vs_bm25.csv', index=False)
display(delta_df.head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
page_delta_summary = delta_df.groupby('run_name')['page_mrr_delta_vs_bm25'].mean().sort_values(ascending=False)
doc_delta_summary = delta_df.groupby('run_name')['doc_mrr_delta_vs_bm25'].mean().sort_values(ascending=False)
page_delta_summary.plot(kind='bar', ax=axes[0], title='Mean Page MRR Delta vs BM25')
doc_delta_summary.plot(kind='bar', ax=axes[1], title='Mean Doc MRR Delta vs BM25', color='tab:orange')
axes[0].axhline(0.0, color='black', linewidth=1)
axes[1].axhline(0.0, color='black', linewidth=1)
axes[0].set_ylabel('delta')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'delta_vs_bm25.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close()

print(f'Saved BM25 deltas to {ANALYSIS_DIR / "delta_vs_bm25.csv"}')
print(f'Saved BM25 delta plot to {ANALYSIS_DIR / "delta_vs_bm25.png"}')


In [ ]:
page_mrr_pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0)
doc_mrr_pivot = per_query_df.pivot(index='uid', columns='run_name', values='doc_mrr').fillna(0)

win_loss_rows = []
for run_name in completed_runs:
    if run_name == 'bm25':
        continue
    page_delta = page_mrr_pivot[run_name] - page_mrr_pivot['bm25']
    doc_delta = doc_mrr_pivot[run_name] - doc_mrr_pivot['bm25']
    win_loss_rows.append({
        'run_name': run_name,
        'page_queries_better_than_bm25': int((page_delta > 0).sum()),
        'page_queries_equal_to_bm25': int((page_delta == 0).sum()),
        'page_queries_worse_than_bm25': int((page_delta < 0).sum()),
        'doc_queries_better_than_bm25': int((doc_delta > 0).sum()),
        'doc_queries_equal_to_bm25': int((doc_delta == 0).sum()),
        'doc_queries_worse_than_bm25': int((doc_delta < 0).sum()),
        'mean_page_mrr_delta_vs_bm25': float(page_delta.mean()),
        'mean_doc_mrr_delta_vs_bm25': float(doc_delta.mean()),
    })

winner_rows = []
for uid in page_mrr_pivot.index:
    best_page_score = page_mrr_pivot.loc[uid].max()
    best_page_runs = [run for run, score in page_mrr_pivot.loc[uid].items() if score == best_page_score]
    for run_name in best_page_runs:
        winner_rows.append({'uid': uid, 'winner_run': run_name})

winner_counts = pd.DataFrame(winner_rows).groupby('winner_run').size().reset_index(name='page_mrr_query_wins')
win_loss_df = pd.DataFrame(win_loss_rows).merge(winner_counts, how='left', left_on='run_name', right_on='winner_run').drop(columns=['winner_run'])
win_loss_df['page_mrr_query_wins'] = win_loss_df['page_mrr_query_wins'].fillna(0).astype(int)
win_loss_df.to_csv(ANALYSIS_DIR / 'win_loss_summary.csv', index=False)
display(win_loss_df.sort_values('mean_page_mrr_delta_vs_bm25', ascending=False))

print(f'Saved win/loss summary to {ANALYSIS_DIR / "win_loss_summary.csv"}')


In [ ]:
crop_compare_rows = []
for base_run, crop_run, label in [
    ('clip_faiss_full', 'clip_faiss_fixed', 'CLIP'),
    ('siglip_faiss_full', 'siglip_faiss_fixed', 'SigLIP'),
]:
    if base_run not in page_mrr_pivot.columns or crop_run not in page_mrr_pivot.columns:
        continue
    for uid in page_mrr_pivot.index:
        crop_compare_rows.append({
            'uid': uid,
            'family': label,
            'full_run': base_run,
            'crop_run': crop_run,
            'page_mrr_full': page_mrr_pivot.loc[uid, base_run],
            'page_mrr_crop': page_mrr_pivot.loc[uid, crop_run],
            'page_mrr_crop_minus_full': page_mrr_pivot.loc[uid, crop_run] - page_mrr_pivot.loc[uid, base_run],
        })

crop_compare_df = pd.DataFrame(crop_compare_rows)
if crop_compare_df.empty:
    print('No full-vs-crop pairs are complete yet.')
else:
    crop_compare_df.to_csv(ANALYSIS_DIR / 'crop_benefit.csv', index=False)
    display(crop_compare_df.head(20))
    fig, axes = plt.subplots(1, len(crop_compare_df['family'].unique()), figsize=(7 * len(crop_compare_df['family'].unique()), 4), squeeze=False)
    for ax, (family, group) in zip(axes[0], crop_compare_df.groupby('family')):
        group = group.sort_values('page_mrr_crop_minus_full', ascending=False)
        ax.bar(group['uid'], group['page_mrr_crop_minus_full'])
        ax.axhline(0.0, color='black', linewidth=1)
        ax.set_title(f'{family}: crop minus full page MRR')
        ax.set_ylabel('delta')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'crop_benefit.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved crop comparison to {ANALYSIS_DIR / "crop_benefit.csv"}')
    print(f'Saved crop comparison plot to {ANALYSIS_DIR / "crop_benefit.png"}')


In [ ]:
def export_pairwise_qualitative(run_name: str, baseline_run: str = 'bm25', top_n_queries: int = 5, per_run_depth: int = 5):
    if run_name not in completed_runs or baseline_run not in completed_runs:
        raise ValueError('Both runs must be completed before exporting qualitative comparisons.')

    page_mrr_pivot = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0)
    delta = (page_mrr_pivot[run_name] - page_mrr_pivot[baseline_run]).sort_values(ascending=False)
    selected_uids = list(delta.head(top_n_queries).index) + list(delta.tail(top_n_queries).index)
    selected_uids = list(dict.fromkeys(selected_uids))

    rows = []
    for compare_run in [baseline_run, run_name]:
        ranked_df = load_ranked_df(compare_run)
        for uid in selected_uids:
            subset = ranked_df.loc[ranked_df['uid'] == uid].sort_values('rank').head(per_run_depth).copy()
            subset['comparison_target'] = run_name
            subset['baseline_run'] = baseline_run
            subset['page_mrr_delta_vs_baseline'] = float(page_mrr_pivot.loc[uid, run_name] - page_mrr_pivot.loc[uid, baseline_run])
            rows.append(subset)

    output_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    output_path = ANALYSIS_DIR / f'qualitative_compare_{run_name}_vs_{baseline_run}.csv'
    output_df.to_csv(output_path, index=False)
    display(output_df[['uid', 'run_name', 'rank', 'doc_id', 'page_num', 'score', 'is_gold_page', 'matched_crop', 'page_mrr_delta_vs_baseline']].head(50))
    print(f'Saved pairwise qualitative export to {output_path}')


export_pairwise_qualitative('clip_faiss_full')


### Query/Page Relevance Visualizations

These cells build explanation-style visuals for completed runs. They are **not true cross-modal attention maps**. Instead, they use:

- **query token ablation**: how much the page similarity score drops when a query term is removed
- **image occlusion heatmaps**: how much the page similarity score drops when parts of the page are masked

This is still a strong qualitative tool for the report, especially for crop-based retrieval.


In [ ]:
import re
from pathlib import Path
from PIL import Image, ImageDraw
from officeqa_retrieval.vision import VisionTextEncoder

EXPLANATION_DIR = ANALYSIS_DIR / 'explanations'
EXPLANATION_DIR.mkdir(parents=True, exist_ok=True)
EXPLANATION_CACHE_DIR = EXPLANATION_DIR / '_occlusion_cache'
EXPLANATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUN_TO_MODEL_KEY = {
    'clip_faiss_full': 'clip',
    'clip_faiss_fixed': 'clip',
    'siglip_faiss_full': 'siglip',
    'siglip_faiss_fixed': 'siglip',
}
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9$%.,/-]+")
_encoder_cache = {}
_ranked_df_cache = {}


def get_encoder_for_run(run_name: str):
    model_key = RUN_TO_MODEL_KEY.get(run_name)
    if model_key is None:
        raise ValueError(f'No multimodal encoder is defined for run {run_name}')
    if model_key not in _encoder_cache:
        _encoder_cache[model_key] = VisionTextEncoder(model_key=model_key, batch_size=8)
    return _encoder_cache[model_key]


def get_ranked_df_cached(run_name: str):
    if run_name not in _ranked_df_cache:
        _ranked_df_cache[run_name] = load_ranked_df(run_name)
    return _ranked_df_cache[run_name]


def score_query_image(query: str, image_path: Path, encoder: VisionTextEncoder) -> float:
    text_embedding = encoder.embed_texts([query])[0]
    image_embedding = encoder.embed_image_paths([image_path])[0]
    return float(np.dot(text_embedding, image_embedding))


def tokenize_query_terms(query: str):
    return TOKEN_PATTERN.findall(query)


def token_importance_df(query: str, image_path: Path, encoder: VisionTextEncoder):
    terms = tokenize_query_terms(query)
    base_score = score_query_image(query, image_path, encoder)
    rows = []
    for index, term in enumerate(terms):
        ablated_terms = terms[:index] + terms[index + 1:]
        if not ablated_terms:
            continue
        ablated_query = ' '.join(ablated_terms)
        ablated_score = score_query_image(ablated_query, image_path, encoder)
        rows.append({
            'term': term,
            'position': index,
            'score_drop': base_score - ablated_score,
            'ablated_score': ablated_score,
        })
    return base_score, pd.DataFrame(rows).sort_values('score_drop', ascending=False)


def occlusion_heatmap(query: str, image_path: Path, encoder: VisionTextEncoder, cache_key: str, grid_size: int = 6):
    base_score = score_query_image(query, image_path, encoder)
    occlusion_dir = EXPLANATION_CACHE_DIR / cache_key
    occlusion_dir.mkdir(parents=True, exist_ok=True)

    with Image.open(image_path) as image:
        image = image.convert('RGB')
        width, height = image.size
        x_edges = np.linspace(0, width, grid_size + 1, dtype=int)
        y_edges = np.linspace(0, height, grid_size + 1, dtype=int)
        occluded_paths = []
        patch_keys = []
        for row in range(grid_size):
            for col in range(grid_size):
                patch_path = occlusion_dir / f'r{row}_c{col}.png'
                if not patch_path.exists():
                    masked = image.copy()
                    draw = ImageDraw.Draw(masked)
                    box = [int(x_edges[col]), int(y_edges[row]), int(x_edges[col + 1]), int(y_edges[row + 1])]
                    draw.rectangle(box, fill=(127, 127, 127))
                    masked.save(patch_path)
                occluded_paths.append(patch_path)
                patch_keys.append((row, col))

    text_embedding = encoder.embed_texts([query])[0]
    masked_embeddings = encoder.embed_image_paths(occluded_paths)
    masked_scores = masked_embeddings @ text_embedding
    heatmap = np.zeros((grid_size, grid_size), dtype=float)
    for (row, col), score in zip(patch_keys, masked_scores, strict=True):
        heatmap[row, col] = base_score - float(score)
    return base_score, heatmap


def export_relevance_panel(run_name: str, uid: str, rank: int = 1, grid_size: int = 6, top_terms: int = 10):
    ranked_df = get_ranked_df_cached(run_name)
    question = question_by_uid[uid]
    candidate = ranked_df.loc[(ranked_df['uid'] == uid) & (ranked_df['rank'] == rank)].copy()
    if candidate.empty:
        raise ValueError(f'Could not find rank {rank} for {uid} in {run_name}')
    row = candidate.iloc[0]
    image_path = RENDER_CACHE / row['doc_id'] / f"page_{int(row['page_num']):04d}.png"
    if not image_path.exists():
        raise FileNotFoundError(f'Missing rendered page image: {image_path}')

    encoder = get_encoder_for_run(run_name)
    base_score, tokens_df = token_importance_df(question.question, image_path, encoder)
    cache_key = f"{run_name}_{uid}_{row['doc_id']}_page_{int(row['page_num']):04d}_g{grid_size}"
    _, heatmap = occlusion_heatmap(question.question, image_path, encoder, cache_key=cache_key, grid_size=grid_size)

    output_dir = EXPLANATION_DIR / run_name
    output_dir.mkdir(parents=True, exist_ok=True)
    tokens_path = output_dir / f'{uid}_token_importance.csv'
    heatmap_path = output_dir / f'{uid}_occlusion_heatmap.csv'
    figure_path = output_dir / f'{uid}_relevance_panel.png'
    tokens_df.to_csv(tokens_path, index=False)
    pd.DataFrame(heatmap).to_csv(heatmap_path, index=False)

    with Image.open(image_path) as image:
        image = image.convert('RGB')
        matched_crop = row.get('matched_crop') if isinstance(row, pd.Series) else None
        crop_image = None
        if isinstance(matched_crop, str) and matched_crop:
            crop_path = RENDER_CACHE / row['doc_id'] / f"page_{int(row['page_num']):04d}_crops" / matched_crop
            if crop_path.exists():
                crop_image = Image.open(crop_path).convert('RGB')

        cols = 4 if crop_image is not None else 3
        fig, axes = plt.subplots(1, cols, figsize=(6 * cols, 6))
        axes = np.atleast_1d(axes)

        axes[0].imshow(image)
        axes[0].set_title(f'{run_name}: retrieved page\nuid={uid} rank={rank}')
        axes[0].axis('off')

        axes[1].imshow(image)
        axes[1].imshow(heatmap, cmap='magma', alpha=0.45, extent=(0, image.size[0], image.size[1], 0), interpolation='nearest')
        axes[1].set_title(f'Occlusion relevance heatmap\nbase score={base_score:.4f}')
        axes[1].axis('off')

        token_plot_index = 2
        if crop_image is not None:
            axes[2].imshow(crop_image)
            axes[2].set_title(f'Matched crop\n{matched_crop}')
            axes[2].axis('off')
            token_plot_index = 3

        top_token_df = tokens_df.head(top_terms).sort_values('score_drop')
        axes[token_plot_index].barh(top_token_df['term'], top_token_df['score_drop'])
        axes[token_plot_index].set_title('Query token importance\n(score drop when removed)')
        axes[token_plot_index].set_xlabel('score drop')

        gold_pages = gold_pages_for_uid(uid)
        fig.suptitle(
            f"Question: {question.question}\nRetrieved page={row['doc_id']}:{int(row['page_num'])} | Gold match={((row['doc_id'], int(row['page_num'])) in gold_pages)}",
            y=1.02,
            fontsize=12,
        )
        plt.tight_layout()
        plt.savefig(figure_path, dpi=200, bbox_inches='tight')
        plt.show()
        plt.close()

        if crop_image is not None:
            crop_image.close()

    print(f'Saved token importance to {tokens_path}')
    print(f'Saved occlusion heatmap grid to {heatmap_path}')
    print(f'Saved explanation panel to {figure_path}')


def choose_sample_uids(run_name: str, baseline_run: str = 'bm25', n_best: int = 2, n_worst: int = 1):
    if run_name not in completed_runs:
        raise ValueError(f'{run_name} is not completed')
    if baseline_run not in completed_runs:
        raise ValueError(f'{baseline_run} is not completed')
    local_page_mrr = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0)
    delta = (local_page_mrr[run_name] - local_page_mrr[baseline_run]).sort_values(ascending=False)
    selected = list(delta.head(n_best).index) + list(delta.tail(n_worst).index)
    return list(dict.fromkeys(selected))


In [ ]:
EXPLANATION_RUN = 'clip_faiss_fixed' if 'clip_faiss_fixed' in completed_runs else ('clip_faiss_full' if 'clip_faiss_full' in completed_runs else completed_runs[-1])
SAMPLE_UIDS = choose_sample_uids(EXPLANATION_RUN, baseline_run='bm25', n_best=2, n_worst=1)
print('Explanation run:', EXPLANATION_RUN)
print('Sample UIDs:', SAMPLE_UIDS)

delta_preview = per_query_df.pivot(index='uid', columns='run_name', values='page_mrr').fillna(0)
if 'bm25' in delta_preview.columns and EXPLANATION_RUN in delta_preview.columns:
    preview_df = (delta_preview[EXPLANATION_RUN] - delta_preview['bm25']).sort_values(ascending=False).reset_index(name='page_mrr_delta_vs_bm25')
    display(preview_df.head(10))
    display(preview_df.tail(10))


In [ ]:
for uid in SAMPLE_UIDS:
    export_relevance_panel(EXPLANATION_RUN, uid, rank=1, grid_size=6, top_terms=10)


In [ ]:
# Optional: explain one specific run/query pair.
# Example:
# export_relevance_panel('clip_faiss_full', 'UID0001', rank=1, grid_size=6, top_terms=12)
pass


In [ ]:
# Optional embedding-space view for one completed full-page run.
import faiss

EMBEDDING_RUN = next((run_name for run_name in ['clip_faiss_full', 'siglip_faiss_full'] if run_name in completed_runs), None)
MAX_POINTS = 2000
RNG = np.random.default_rng(0)

if EMBEDDING_RUN is None:
    print('No completed full-page FAISS run is available yet for embedding visualization.')
else:
    index_dir = RESULTS_DIR.parent.parent / 'artifacts' / RUN_TAG / EMBEDDING_RUN
    if not (index_dir / 'index.faiss').exists():
        print(f'Could not find index directory for {EMBEDDING_RUN}: {index_dir}')
    else:
        index = faiss.read_index(str(index_dir / 'index.faiss'))
        metadata_rows = [json.loads(line) for line in (index_dir / 'metadata.jsonl').open('r', encoding='utf-8')]
        sample_size = min(MAX_POINTS, index.ntotal)
        sample_ids = np.sort(RNG.choice(index.ntotal, size=sample_size, replace=False))
        vectors = np.vstack([index.reconstruct(int(i)) for i in sample_ids])
        metadata_sample = [metadata_rows[int(i)] for i in sample_ids]

        centered = vectors - vectors.mean(axis=0, keepdims=True)
        _, _, vt = np.linalg.svd(centered, full_matrices=False)
        coords = centered @ vt[:2].T
        years = [row['doc_id'].split('_')[2] if len(row['doc_id'].split('_')) >= 4 else 'unknown' for row in metadata_sample]
        year_codes = pd.Categorical(years)

        plt.figure(figsize=(7, 6))
        scatter = plt.scatter(coords[:, 0], coords[:, 1], c=year_codes.codes, s=10, alpha=0.7, cmap='tab20')
        plt.title(f'Embedding PCA sample: {EMBEDDING_RUN}')
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.tight_layout()
        plt.savefig(ANALYSIS_DIR / f'embedding_pca_{EMBEDDING_RUN}.png', dpi=200, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f'Saved embedding PCA plot to {ANALYSIS_DIR / f"embedding_pca_{EMBEDDING_RUN}.png"}')


In [ ]:
import json
from pathlib import Path
from PIL import Image
from IPython.display import display

TARGET_UID = questions_df['uid'].iloc[0]
TARGET_RUN = 'clip_faiss_fixed'
TOP_N = 3

question_row = questions_df.loc[questions_df['uid'] == TARGET_UID].iloc[0]
print('Question UID:', TARGET_UID)
print('Question:', question_row['question'])
print('Gold source_files:', question_row['source_files'])
print('Gold source_docs:', question_row['source_docs'])

rows = []
for ranked_path in sorted(RESULTS_DIR.glob('*/ranked_pages.jsonl')):
    run_name = ranked_path.parent.name
    with ranked_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            if row['uid'] == TARGET_UID and row['rank'] <= 10:
                row['run_name'] = run_name
                rows.append(row)

qual_df = pd.DataFrame(rows).sort_values(['run_name', 'rank'])
qual_path = ANALYSIS_DIR / f'qualitative_{TARGET_UID}.csv'
qual_df.to_csv(qual_path, index=False)
display(qual_df[['run_name', 'rank', 'doc_id', 'page_num', 'score', 'bm25_score', 'matched_crop']])
print(f'Saved qualitative CSV to {qual_path}')

selected = qual_df.loc[qual_df['run_name'] == TARGET_RUN].head(TOP_N)
for _, row in selected.iterrows():
    page_num = int(row['page_num'])
    page_image_path = RENDER_CACHE / row['doc_id'] / f'page_{page_num:04d}.png'
    print(f"\n{TARGET_RUN} rank={row['rank']} doc={row['doc_id']} page={page_num} score={row['score']:.4f}")
    if page_image_path.exists():
        display(Image.open(page_image_path))
    matched_crop = row.get('matched_crop')
    if isinstance(matched_crop, str) and matched_crop:
        crop_path = RENDER_CACHE / row['doc_id'] / f'page_{page_num:04d}_crops' / matched_crop
        if crop_path.exists():
            print(f'Matched crop: {matched_crop}')
            display(Image.open(crop_path))
